# Learnable PPR Subgraph Selection 

**Phase 1**: Architecture search learns per-edge `(teleport_u, teleport_v)` via bi-level optimization on the full graph.\
**Phase 2**: Fine-tune a GNN on extracted subgraphs using the learned configurations.

Inspired by [PS2 (WSDM'23)](https://github.com/qiaoyu-tan/PS2) which learns per-edge `(k_u, k_v)` for k-hop.

In [ ]:
# === CONFIGURABLE SEARCH SPACE ===
# Edit these values to change the PPR search space.
# Number of configurations = len(TELEPORT_VALUES)^2

TELEPORT_VALUES = [0.50, 0.85, 0.95]   # PPR teleport probabilities
ALPHA = 0.5                              # Fixed alpha for combining PPR_u + PPR_v
PPR_EPSILON = 1e-3                       # Approximate PPR precision (Phase 2 sweep cut)
PPR_WINDOW  = 10                         # Conductance sweep cut window

DATASETS = ['FB15K237']                  # FB15K237, WN18RR, NELL-995
ENCODERS = ['GCN', 'SAGE', 'GAT']       # All 3 encoders per run

# Hyperparameters
FEATURE_DIM = 128
HIDDEN_CHANNELS = 256
NUM_LAYERS = 3
DROPOUT = 0.3

# Phase 1: Architecture Search (temperature annealing controls horizon, no early stopping)
SEARCH_EPOCHS = 50
SEARCH_BATCH_SIZE = 1024
SEARCH_LR = 0.01
TEMPERATURE_START = 1.0                  # Annealing: start (soft exploration)
TEMPERATURE_END = 0.2                    # Annealing: end (sharp enough for 9 configs)
EDGES_PER_SEARCH_EPOCH = 100_000         # Subsample search edges (None = all)
FIRST_ORDER = True                       # Skip HVP (first-order DARTS)
SEARCH_VAL_EVERY = 5                     # Full validation every N search epochs
USE_AMP = True                           # Mixed precision (float16 forward/matmul)

# Phase 2: Fine-tuning
FINETUNE_EPOCHS = 500
FINETUNE_BATCH_SIZE = 8192
FINETUNE_LR = 0.005
FINETUNE_PATIENCE = 30
MAX_SUBGRAPHS_PER_FORWARD = 256
EDGES_PER_EPOCH = 100000           # subsample training edges per epoch (None = use all)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Search space: {len(TELEPORT_VALUES)}x{len(TELEPORT_VALUES)} = {len(TELEPORT_VALUES)**2} configs')
print(f'Encoders: {ENCODERS}')
print(f'Temperature annealing: {TEMPERATURE_START} -> {TEMPERATURE_END}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import json, os, time, copy
import pandas as pd

SEED = 42
import torch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
import random
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sns.set_theme(style='whitegrid', palette='muted')

from subgrapher.benchmark.data_prep import prepare_link_prediction_data
from subgrapher.utils.models import GCN, SAGE, GAT, LinkPredictor
from subgrapher.benchmark_learnable_ppr.multi_scale_ppr import MultiScalePPR
from subgrapher.benchmark_learnable_ppr.autolink_ppr import AutoLinkPPR
from subgrapher.benchmark_learnable_ppr.search_net import PPRSearchNet
from subgrapher.benchmark_learnable_ppr.searcher import ArchitectureSearcher
from subgrapher.benchmark_learnable_ppr.finetuner import finetune_on_subgraphs, LearnablePPRExtractor, SubgraphCache, build_or_load_cache
from subgrapher.benchmark_learnable_ppr.evaluator import evaluate_learnable_ppr, evaluate_learnable_ppr_fullgraph, print_evaluation_results
from subgrapher.benchmark_learnable_ppr.artifacts import save_learnable_ppr_experiment


def ensure_train_only_edges(data, dataset_dict):
    """Point data.edge_index at the train-only edge set (idempotent).

    prepare_link_prediction_data returns a Data whose edge_index contains ALL
    positives (train + val + test). Extracting a subgraph around a val/test edge
    (u, v) with the full graph would include (u, v) itself and leak the target
    into message passing. Every section that uses data.edge_index for subgraph
    extraction should call this first. Safe to call multiple times.
    """
    if not getattr(data, '_edge_index_train_only', False):
        data._orig_edge_index = data.edge_index
        data.edge_index = dataset_dict['train_edge_index']
        data._edge_index_train_only = True
        print(f'  [train-only edges] swapped: '
              f"{data._orig_edge_index.size(1):,} -> {data.edge_index.size(1):,}")
    return data


## 1. Data Loading and Multi-Scale PPR

In [ ]:
DATASET_PATHS = {
    'FB15K237': 'data/FB15K237/train.txt',
    'WN18RR': 'data/WN18RR/train.txt',
    'NELL-995': 'data/NELL-995/train.txt',
}

all_results = {}

for dataset_name in DATASETS:
    print(f'\n{"=" * 60}')
    print(f'Dataset: {dataset_name}')
    print(f'{"=" * 60}')

    dataset_dict = prepare_link_prediction_data(
        DATASET_PATHS[dataset_name],
        feature_method='random', feature_dim=FEATURE_DIM)

    data = dataset_dict['data']
    split_edge = dataset_dict['split_edge']

    print(f'Nodes: {data.num_nodes}, Edges: {data.edge_index.size(1)}')
    print(f'Train: {split_edge["train"]["source_node"].size(0)}, '
          f'Val: {split_edge["valid"]["source_node"].size(0)}, '
          f'Test: {split_edge["test"]["source_node"].size(0)}')

    # Fix val/test leakage: use train-only edges for message passing.
    ensure_train_only_edges(data, dataset_dict)

    print(f'\nLoading multi-scale PPR (teleport={TELEPORT_VALUES})...')
    multi_scale_ppr = MultiScalePPR(
        dataset_name, data, teleport_values=TELEPORT_VALUES, device=DEVICE)
    print(multi_scale_ppr)

    all_results[dataset_name] = {
        'data': data, 'split_edge': split_edge,
        'dataset_dict': dataset_dict,
        'multi_scale_ppr': multi_scale_ppr, 'experiments': {},
    }

## 2. Phase 1: Architecture Search

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])
    num_configs = msp.num_configs

    for enc_type in ENCODERS:
        print(f'\n{"=" * 60}')
        print(f'Phase 1: {dataset_name} / {enc_type}')
        print(f'{"=" * 60}')

        model = AutoLinkPPR(
            in_channels=FEATURE_DIM, hidden_channels=HIDDEN_CHANNELS,
            num_layers=NUM_LAYERS, dropout=DROPOUT,
            gnn_type=enc_type, num_configs=num_configs)

        arch_net = PPRSearchNet(
            in_channels=HIDDEN_CHANNELS, hidden_channels=256,
            num_layers=3, temperature=TEMPERATURE_START)

        searcher = ArchitectureSearcher(
            model, arch_net, msp, data, split_edge,
            device=DEVICE, lr=SEARCH_LR, lr_arch=SEARCH_LR,
            temperature_start=TEMPERATURE_START,
            temperature_end=TEMPERATURE_END,
            edges_per_search_epoch=EDGES_PER_SEARCH_EPOCH,
            first_order=FIRST_ORDER)

        search_history = searcher.search(
            epochs=SEARCH_EPOCHS, batch_size=SEARCH_BATCH_SIZE,
            verbose=True, val_every=SEARCH_VAL_EVERY, use_amp=USE_AMP)

        print(f'Search done in {search_history["total_time"]:.1f}s')

        r['experiments'][enc_type] = {
            'search_history': search_history,
            'model': model, 'arch_net': arch_net,
        }

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        searcher_ref = ArchitectureSearcher(
            exp['model'], exp['arch_net'], msp, data, split_edge,
            device=DEVICE, first_order=FIRST_ORDER)

        print(f'Extracting configs: {dataset_name} / {enc_type}')
        train_configs, train_counts = searcher_ref.get_edge_configs('train')
        val_configs, val_counts = searcher_ref.get_edge_configs('valid')
        test_configs, test_counts = searcher_ref.get_edge_configs('test')

        exp.update({
            'train_configs': train_configs, 'val_configs': val_configs,
            'test_configs': test_configs,
            'train_counts': train_counts, 'val_counts': val_counts,
            'test_counts': test_counts,
        })
        print(f'  Train config counts: {train_counts.tolist()}')

        p1_dir = f'results/phase1-checkpoint/{dataset_name}/{enc_type}'
        os.makedirs(p1_dir, exist_ok=True)
        search_history = exp['search_history']
        p1_save = {
            'search_loss': search_history['search_loss'],
            'val_loss': [v for v in search_history['val_loss'] if v is not None],
            'best_val_loss': search_history['best_val_loss'],
            'best_epoch': search_history['best_epoch'],
            'total_time': search_history['total_time'],
            'temperature': search_history['temperature'],
            'arch_entropy': search_history['arch_entropy'],
            'softmax_mass_top1': search_history['softmax_mass_top1'],
            'config_distributions': search_history['config_distributions'],
            'train_counts': train_counts.tolist(),
            'val_counts': val_counts.tolist(),
            'test_counts': test_counts.tolist(),
        }
        with open(os.path.join(p1_dir, 'search_history.json'), 'w') as f:
            json.dump(p1_save, f, indent=2)
        torch.save({
            'model_state': exp['model'].state_dict(),
            'arch_net_state': exp['arch_net'].state_dict(),
            'train_configs': train_configs,
            'val_configs': val_configs,
            'test_configs': test_configs,
        }, os.path.join(p1_dir, 'phase1_checkpoint.pt'))
        print(f'  Checkpoint saved: {p1_dir}')

## 3. Config Distribution Heatmap

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    msp = r['multi_scale_ppr']; S = msp.num_scales

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, (split, counts) in zip(axes, [
            ('Train', exp['train_counts']),
            ('Valid', exp['val_counts']),
            ('Test', exp['test_counts']),
        ]):
            matrix = counts.numpy().reshape(S, S).astype(float)
            total = matrix.sum()
            pct = 100 * matrix / max(total, 1)
            annot = np.array([[f'{int(matrix[i,j])}\n({pct[i,j]:.1f}%)'
                               for j in range(S)] for i in range(S)])
            sns.heatmap(pct, annot=annot, fmt='',
                        xticklabels=[f'{t}' for t in msp.teleport_values],
                        yticklabels=[f'{t}' for t in msp.teleport_values],
                        cmap='YlOrRd', vmin=0, ax=ax)
            ax.set_xlabel('teleport_v'); ax.set_ylabel('teleport_u')
            ax.set_title(f'{split} ({int(total)} edges)')

        fig.suptitle(f'{dataset_name} / {enc_type} -- Learned Config Distribution',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()

## 4. Architecture Search Training Curves

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    for enc_type in ENCODERS:
        hist = r['experiments'][enc_type]['search_history']
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        epochs_list = range(1, len(hist['search_loss']) + 1)
        sns.lineplot(x=list(epochs_list), y=hist['search_loss'], ax=ax1, label='Train Loss')
        val_epochs = [e for e, v in zip(epochs_list, hist['val_loss']) if v is not None]
        val_vals = [v for v in hist['val_loss'] if v is not None]
        if val_vals:
            sns.lineplot(x=val_epochs, y=val_vals, ax=ax1, label='Val Loss')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
        ax1.set_title('Phase 1: Search Loss')
        ax1.axvline(hist['best_epoch'], color='red', linestyle='--', alpha=0.5,
                    label=f'Best ({hist["best_epoch"]})')
        ax1.legend()

        ax2.text(0.5, 0.5,
                 f'Best val loss: {hist["best_val_loss"]:.4f}\n'
                 f'Best epoch: {hist["best_epoch"]}\n'
                 f'Total time: {hist["total_time"]:.1f}s',
                 transform=ax2.transAxes, ha='center', va='center', fontsize=14)
        ax2.set_title('Search Summary'); ax2.axis('off')
        fig.suptitle(f'{dataset_name} / {enc_type} -- Architecture Search',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()

## 4a. Phase 1 Diagnostic Dashboard

Combined view: search loss, val loss, arch entropy, temperature, and config collapse detection.
NaN regions are highlighted in red.

In [ ]:
import math

for dataset_name in DATASETS:
    r = all_results[dataset_name]
    for enc_type in ENCODERS:
        hist = r['experiments'][enc_type]['search_history']
        if not hist.get('temperature'): continue
        n_ep = len(hist['temperature'])
        epochs_list = list(range(1, n_ep + 1))

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))

        # -- (0,0) Search + Val loss --
        ax = axes[0, 0]
        losses = hist['search_loss']
        nan_mask = [math.isnan(v) if isinstance(v, float) else False for v in losses]
        clean_losses = [v if not m else None for v, m in zip(losses, nan_mask)]
        ax.plot(epochs_list, clean_losses, label='Train Loss')
        if any(nan_mask):
            nan_epochs = [e for e, m in zip(epochs_list, nan_mask) if m]
            for ne in nan_epochs:
                ax.axvspan(ne - 0.5, ne + 0.5, color='red', alpha=0.15)
            ax.text(0.02, 0.95, f'NaN in {sum(nan_mask)}/{n_ep} epochs',
                    transform=ax.transAxes, color='red', fontsize=10, va='top',
                    fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightyellow'))
        val_epochs = [e for e, v in zip(epochs_list, hist['val_loss']) if v is not None]
        val_vals = [v for v in hist['val_loss'] if v is not None]
        if val_vals:
            ax.plot(val_epochs, val_vals, marker='.', label='Val Loss')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.set_title('Search Loss'); ax.legend()

        # -- (0,1) Temperature + Entropy --
        ax1 = axes[0, 1]
        ax1.plot(epochs_list, hist['temperature'], color='tab:blue', label='Temperature')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Temperature', color='tab:blue')
        ax1.tick_params(axis='y', labelcolor='tab:blue')
        ax2 = ax1.twinx()
        ent = hist['arch_entropy']
        ent_clean = [v if not (isinstance(v, float) and math.isnan(v)) else None for v in ent]
        ax2.plot(epochs_list, ent_clean, color='tab:orange', label='Entropy')
        if hist.get('softmax_mass_top1'):
            mass = hist['softmax_mass_top1']
            mass_clean = [v if not (isinstance(v, float) and math.isnan(v)) else None for v in mass]
            ax2.plot(epochs_list, mass_clean, color='tab:green', linestyle='--', label='Top-1 mass')
        ax2.set_ylabel('Entropy / Mass', color='tab:orange')
        ax2.tick_params(axis='y', labelcolor='tab:orange')
        lines1, lab1 = ax1.get_legend_handles_labels()
        lines2, lab2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, lab1 + lab2, loc='center right')
        ax1.set_title('Temperature + Entropy')

        # -- (1,0) Config distribution over time --
        ax = axes[1, 0]
        cdists = hist.get('config_distributions', [])
        if cdists:
            msp = r['multi_scale_ppr']
            labels = [f'({tu},{tv})' for tu, tv in msp.config_labels]
            snap_epochs = [d['epoch'] for d in cdists]
            counts_matrix = np.array([d['train_counts'] for d in cdists], dtype=float)
            totals = counts_matrix.sum(axis=1, keepdims=True).clip(min=1)
            pcts = 100 * counts_matrix / totals
            bottom = np.zeros(len(snap_epochs))
            for ci in range(pcts.shape[1]):
                ax.bar(snap_epochs, pcts[:, ci], bottom=bottom, label=labels[ci], width=2)
                bottom += pcts[:, ci]
            ax.set_xlabel('Epoch'); ax.set_ylabel('% of edges')
            ax.legend(fontsize=7, ncol=3, loc='upper right')
        ax.set_title('Config Distribution Over Time')

        # -- (1,1) Degenerate config warning --
        ax = axes[1, 1]
        exp = r['experiments'][enc_type]
        total_train = exp['train_counts'].sum().item()
        max_count = exp['train_counts'].max().item()
        max_pct = 100 * max_count / max(total_train, 1)
        dominant_idx = exp['train_counts'].argmax().item()
        msp = r['multi_scale_ppr']
        dom_label = f'({msp.config_labels[dominant_idx][0]},{msp.config_labels[dominant_idx][1]})'

        status_lines = [
            f'Dominant config: {dom_label} ({max_pct:.1f}%)',
            f'Total time: {hist["total_time"]:.0f}s',
            f'Best val loss: {hist["best_val_loss"]:.4f}',
            f'Best epoch: {hist["best_epoch"]}',
            f'NaN epochs: {sum(nan_mask)}/{n_ep}',
        ]
        if max_pct > 95:
            status_lines.insert(0, 'WARNING: CONFIG COLLAPSE (>95% single config)')
            ax.set_facecolor('#fff0f0')
        elif all(nan_mask):
            status_lines.insert(0, 'WARNING: ALL LOSSES NaN -- search failed')
            ax.set_facecolor('#fff0f0')
        else:
            status_lines.insert(0, 'Status: OK')
            ax.set_facecolor('#f0fff0')
        ax.text(0.5, 0.5, '\n'.join(status_lines),
                transform=ax.transAxes, ha='center', va='center', fontsize=12,
                fontfamily='monospace')
        ax.set_title('Phase 1 Summary'); ax.axis('off')

        fig.suptitle(f'{dataset_name} / {enc_type} -- Phase 1 Diagnostics',
                     fontsize=15, fontweight='bold')
        plt.tight_layout(); plt.show()

## 4b. Embedding Norm Distribution

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    has_data = any(r['experiments'].get(e, {}).get('search_history', {}).get('embedding_norm_mean')
                   for e in ENCODERS)
    if not has_data: continue

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for enc_type in ENCODERS:
        hist = r['experiments'][enc_type]['search_history']
        if not hist.get('embedding_norm_mean'): continue
        epochs_list = range(1, len(hist['embedding_norm_mean']) + 1)
        ax1.plot(list(epochs_list), hist['embedding_norm_mean'], label=enc_type)
        ax2.plot(list(epochs_list), hist['embedding_norm_max'], label=enc_type)

    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Mean ||h||')
    ax1.set_title('Mean Embedding Norm'); ax1.legend()
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Max ||h||')
    ax2.set_title('Max Embedding Norm'); ax2.legend()
    fig.suptitle(f'{dataset_name} -- Embedding Norms (after L2 norm)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

## 5. Phase 2: Fine-tune on Extracted Subgraphs

### 5a. Build / load subgraph caches (one-time cost, saved to disk)

In [ ]:
def _create_encoder(enc_type, in_ch, hid, n_layers, dropout):
    if enc_type == 'GCN': return GCN(in_ch, hid, hid, n_layers, dropout)
    elif enc_type == 'SAGE': return SAGE(in_ch, hid, hid, n_layers, dropout)
    elif enc_type == 'GAT': return GAT(in_ch, hid, hid, n_layers, dropout, heads=4)
    raise ValueError(enc_type)

for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        exp['ft_encoder'] = _create_encoder(enc_type, FEATURE_DIM, HIDDEN_CHANNELS, NUM_LAYERS, DROPOUT)
        exp['ft_predictor'] = LinkPredictor(HIDDEN_CHANNELS, HIDDEN_CHANNELS, 1, NUM_LAYERS, DROPOUT)
        exp['ckpt_dir'] = f'checkpoints/learnable-ppr/{dataset_name}/{enc_type}'
        exp['cache_dir'] = f'cache/learnable-ppr/{dataset_name}/{enc_type}'

        print(f'\n[Cache] {dataset_name} / {enc_type} (eps={PPR_EPSILON}, window={PPR_WINDOW})')
        extractor = LearnablePPRExtractor(data, msp, exp['train_configs'], alpha=ALPHA, epsilon=PPR_EPSILON, window=PPR_WINDOW)
        build_or_load_cache(extractor, split_edge, 'train', exp['cache_dir'])

print('Caches ready.')

### 5b. Fine-tune (resumes from checkpoint if interrupted)

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])

    for enc_type in ENCODERS:
        print(f'\nPhase 2: {dataset_name} / {enc_type} (eps={PPR_EPSILON}, window={PPR_WINDOW})')
        exp = r['experiments'][enc_type]

        ft_history = finetune_on_subgraphs(
            exp['ft_encoder'], exp['ft_predictor'], data, split_edge,
            msp, exp['train_configs'],
            alpha=ALPHA, epsilon=PPR_EPSILON, window=PPR_WINDOW,
            epochs=FINETUNE_EPOCHS, batch_size=FINETUNE_BATCH_SIZE,
            lr=FINETUNE_LR, eval_steps=5, device=DEVICE,
            verbose=True, patience=FINETUNE_PATIENCE,
            max_subgraphs_per_forward=MAX_SUBGRAPHS_PER_FORWARD,
            checkpoint_dir=exp['ckpt_dir'],
            cache_dir=exp['cache_dir'],
            edges_per_epoch=EDGES_PER_EPOCH,
            val_config_indices=exp['val_configs'])

        exp['ft_history'] = ft_history
        print(f'Best MRR: {ft_history["best_val_mrr"]:.4f} at epoch {ft_history["best_epoch"]}')

## 6. Test Evaluation

Two evaluation modes:
- **Subgraph eval**: per-edge subgraph encoding (matches training). Inflated by 0.1 fallback.
- **Full-graph eval**: full-graph encoding, all negatives scored by model. **Use this for comparisons.**

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]

        # Subgraph eval (matches training, kept for reference)
        cache_dir = f'cache/learnable-ppr/{dataset_name}/{enc_type}'
        test_sub = evaluate_learnable_ppr(
            exp['ft_encoder'], exp['ft_predictor'],
            data, split_edge, msp, exp['test_configs'],
            split='test', alpha=ALPHA, epsilon=PPR_EPSILON, window=PPR_WINDOW, device=DEVICE,
            cache_dir=cache_dir)
        exp['test_results_subgraph'] = test_sub

        # Full-graph eval (fair, comparable to baselines)
        test_fg = evaluate_learnable_ppr_fullgraph(
            exp['ft_encoder'], exp['ft_predictor'],
            data, split_edge,
            split='test', device=DEVICE)
        exp['test_results_fullgraph'] = test_fg
        exp['test_results'] = test_fg  # default for comparisons

        print(f'\n{dataset_name} / {enc_type}')
        print('--- Full-graph eval (for comparison) ---')
        print_evaluation_results(test_fg, 'test')
        print('--- Subgraph eval (reference only) ---')
        print_evaluation_results(test_sub, 'test (subgraph)')

## 7. Fine-Tuning Training Curves (MRR + full val metrics)

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        hist = exp.get('ft_history', {})
        if not hist: continue

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        epochs_list = range(1, len(hist['train_loss']) + 1)
        sns.lineplot(x=list(epochs_list), y=hist['train_loss'], ax=axes[0], label='Train Loss')
        axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
        axes[0].set_title('Phase 2: Fine-tune Loss')

        if hist['val_results']:
            val_mrrs = [vr['mrr'] for vr in hist['val_results']]
            eval_epochs = list(range(5, 5 * len(val_mrrs) + 1, 5))
            sns.lineplot(x=eval_epochs, y=val_mrrs, ax=axes[1], marker='o', label='MRR')
            axes[1].axhline(hist['best_val_mrr'], color='red', linestyle='--', alpha=0.5,
                            label=f'Best: {hist["best_val_mrr"]:.4f}')
            axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MRR')
            axes[1].set_title('Validation MRR'); axes[1].legend()

            val_aucs = [vr.get('auc', 0) for vr in hist['val_results']]
            val_aps = [vr.get('ap', 0) for vr in hist['val_results']]
            if any(val_aucs):
                sns.lineplot(x=eval_epochs, y=val_aucs, ax=axes[2], label='AUC')
            if any(val_aps):
                sns.lineplot(x=eval_epochs, y=val_aps, ax=axes[2], label='AP')
            axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Score')
            axes[2].set_title('Val AUC / AP'); axes[2].legend()

        fig.suptitle(f'{dataset_name} / {enc_type} -- Fine-tuning', fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()

## 8. Cross-Encoder Comparison + Static PPR Baselines

In [ ]:
def load_baseline_results():
    baselines = []
    base_dir = 'results/benchmark-ppr'
    if not os.path.exists(base_dir): return baselines
    for ds in os.listdir(base_dir):
        ds_dir = os.path.join(base_dir, ds)
        if not os.path.isdir(ds_dir): continue
        for exp_name in os.listdir(ds_dir):
            rp = os.path.join(ds_dir, exp_name, 'full_results.json')
            if os.path.exists(rp):
                with open(rp) as f:
                    baselines.append(json.load(f))
    return baselines

baselines = load_baseline_results()

rows = []
for b in baselines:
    if b['dataset'] in DATASETS and b['encoder'] in ENCODERS:
        label = f'Static PPR (a={b.get("ppr_alpha", "?")}, e={b.get("ppr_epsilon", b.get("top_k", "?"))})'
        rows.append({'Dataset': b['dataset'], 'Encoder': b['encoder'],
                     'Method': label,
                     'MRR': b['test_results']['mrr'],
                     'AUC': b['test_results']['auc']})

for ds in DATASETS:
    r = all_results.get(ds, {})
    for enc in ENCODERS:
        exp = r.get('experiments', {}).get(enc, {})
        tr = exp.get('test_results')
        if tr:
            rows.append({'Dataset': ds, 'Encoder': enc, 'Method': 'Learnable PPR',
                         'MRR': tr['mrr'], 'AUC': tr['auc']})

if rows:
    df = pd.DataFrame(rows)
    for metric in ['MRR', 'AUC']:
        fig, ax = plt.subplots(figsize=(max(10, 2.5 * len(ENCODERS)), 5))
        sns.barplot(data=df, x='Encoder', y=metric, hue='Method', ax=ax)
        ax.set_title(f'Test {metric} -- Cross-Encoder Comparison')
        ax.legend(loc='lower right')
        plt.tight_layout(); plt.show()
    display(df.pivot_table(index=['Dataset', 'Encoder'], columns='Method', values=['MRR', 'AUC']))
else:
    print('No results to compare yet.')

### 8a. Phase 1 Wall-Clock per Encoder

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    enc_names = []; search_times = []
    for enc_type in ENCODERS:
        exp = r['experiments'].get(enc_type)
        if exp:
            enc_names.append(enc_type)
            search_times.append(exp['search_history']['total_time'])

    if enc_names:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh(enc_names, [t / 60 for t in search_times])
        ax.set_xlabel('Phase 1 Time (minutes)')
        ax.set_title(f'{dataset_name} -- Search Time by Encoder')
        for i, t in enumerate(search_times):
            ax.text(t / 60 + 0.5, i, f'{t:.0f}s', va='center')
        plt.tight_layout(); plt.show()

## 9. Per-Edge Config Analysis by Node Degree

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])

    degrees = torch.zeros(data.num_nodes, dtype=torch.long)
    for i in range(data.edge_index.size(1)):
        degrees[data.edge_index[0, i]] += 1

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        train_src = r['split_edge']['train']['source_node']
        configs = exp['train_configs']

        src_degrees = degrees[train_src].numpy()
        config_labels = [f'({tu},{tv})' for tu, tv in msp.config_labels]
        config_names = np.array([config_labels[c.item()] for c in configs])

        degree_bins = np.digitize(src_degrees, bins=[0, 5, 20, 50, 200, 1000])
        bin_labels = ['1-5', '6-20', '21-50', '51-200', '200+']

        df_deg = pd.DataFrame({
            'Degree Bin': [bin_labels[min(b-1, len(bin_labels)-1)] for b in degree_bins],
            'Config': config_names,
        })

        fig, ax = plt.subplots(figsize=(12, 5))
        sns.countplot(data=df_deg, x='Degree Bin', hue='Config', order=bin_labels, ax=ax)
        ax.set_title(f'{dataset_name} / {enc_type} -- Config by Source Node Degree')
        ax.set_ylabel('Count')
        ax.legend(title='(teleport_u, teleport_v)', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout(); plt.show()

## 10. t-SNE of Node Embeddings Colored by Config Preference

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        model = exp['model']; model.eval()

        with torch.no_grad():
            data_dev = data.to(DEVICE)
            H = model(data_dev.x, data_dev.edge_index).cpu().numpy()

        train_src = r['split_edge']['train']['source_node']
        configs = exp['train_configs']
        node_config_votes = torch.zeros(data.num_nodes, msp.num_configs)
        for s, c in zip(train_src, configs):
            node_config_votes[s, c] += 1

        dominant_config = node_config_votes.argmax(dim=1).numpy()
        has_votes = node_config_votes.sum(dim=1) > 0

        mask = has_votes.numpy()
        indices = np.where(mask)[0]
        if len(indices) > 3000:
            indices = np.random.choice(indices, 3000, replace=False)

        H_sub = H[indices]
        labels_sub = dominant_config[indices]

        print(f't-SNE for {dataset_name}/{enc_type} ({len(indices)} nodes)...')
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        emb_2d = tsne.fit_transform(H_sub)

        config_labels = [f'({tu},{tv})' for tu, tv in msp.config_labels]
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=labels_sub,
                             cmap='tab10', s=10, alpha=0.6)
        cbar = plt.colorbar(scatter, ax=ax, ticks=range(msp.num_configs))
        cbar.ax.set_yticklabels(config_labels)
        ax.set_title(f'{dataset_name} / {enc_type} -- t-SNE by dominant PPR config')
        ax.axis('off')
        plt.tight_layout(); plt.show()

## 11. Subgraph Size Comparison

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]
    data = r['data']; split_edge = r['split_edge']; msp = r['multi_scale_ppr']
    ensure_train_only_edges(data, r['dataset_dict'])

    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        extractor = LearnablePPRExtractor(
            data, msp, exp['train_configs'], alpha=ALPHA, epsilon=PPR_EPSILON, window=PPR_WINDOW)

        src = split_edge['train']['source_node']
        dst = split_edge['train']['target_node']
        n_sample = min(500, src.size(0))
        sample_idx = torch.randperm(src.size(0))[:n_sample]

        sizes_learned = []; configs_used = []
        for i in sample_idx:
            u, v = src[i].item(), dst[i].item()
            _, sel, meta = extractor.extract_subgraph(u, v, i.item())
            sizes_learned.append(meta['num_nodes_selected'])
            configs_used.append(meta['config_idx'])

        config_labels = [f'({tu},{tv})' for tu, tv in msp.config_labels]
        df_sz = pd.DataFrame({
            'Subgraph Size': sizes_learned,
            'Config': [config_labels[c] for c in configs_used],
        })

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        sns.histplot(sizes_learned, bins=30, ax=ax1, kde=True)
        ax1.axvline(np.mean(sizes_learned), color='red', linestyle='--',
                    label=f'Mean: {np.mean(sizes_learned):.0f}')
        ax1.set_xlabel('Subgraph Size (nodes)'); ax1.set_title('Size Distribution'); ax1.legend()

        sns.boxplot(data=df_sz, x='Config', y='Subgraph Size', ax=ax2)
        ax2.set_title('Size by Config'); ax2.tick_params(axis='x', rotation=45)

        fig.suptitle(f'{dataset_name} / {enc_type} -- Subgraph Sizes (sweep cut)',
                     fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()

## 12. Save Results

In [ ]:
for dataset_name in DATASETS:
    r = all_results[dataset_name]; msp = r['multi_scale_ppr']
    for enc_type in ENCODERS:
        exp = r['experiments'][enc_type]
        tr = exp.get('test_results')
        if not tr: continue

        save_dir = f'results/benchmark-learnable-ppr/{dataset_name}/{enc_type}'
        os.makedirs(save_dir, exist_ok=True)

        tr_fg = exp.get('test_results_fullgraph', tr)
        tr_sub = exp.get('test_results_subgraph', {})
        search_time = exp['search_history']['total_time']
        finetune_time = exp['ft_history'].get('total_time', 0)
        run_id = time.strftime('%Y%m%d_%H%M%S')
        result_json = {
            'dataset': dataset_name, 'encoder': enc_type,
            'teleport_values': TELEPORT_VALUES, 'alpha': ALPHA,
            'ppr_epsilon': PPR_EPSILON, 'ppr_window': PPR_WINDOW,
            'test_results': {k: float(v) for k, v in tr_fg.items()},
            'test_results_subgraph': {k: float(v) for k, v in tr_sub.items()} if tr_sub else {},
            'test_results_fullgraph': {k: float(v) for k, v in tr_fg.items()},
            'eval_mode_for_comparison': 'fullgraph',
            'config_distribution': {
                'train': exp['train_counts'].tolist(),
                'val': exp['val_counts'].tolist(),
                'test': exp['test_counts'].tolist(),
                'labels': [f'({tu},{tv})' for tu, tv in msp.config_labels],
            },
            'search_time': search_time,
            'finetune_time': finetune_time,
            'train_time': search_time + finetune_time,
            'finetune_best_mrr': exp['ft_history']['best_val_mrr'],
            'finetune_best_epoch': exp['ft_history']['best_epoch'],
            'temperature_start': TEMPERATURE_START,
            'temperature_end': TEMPERATURE_END,
            'seed': SEED,
            'run_id': run_id,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        }

        run_dir = os.path.join(save_dir, 'runs', run_id)
        os.makedirs(run_dir, exist_ok=True)
        with open(os.path.join(run_dir, 'full_results.json'), 'w') as f:
            json.dump(result_json, f, indent=2)
        with open(os.path.join(save_dir, 'full_results.json'), 'w') as f:
            json.dump(result_json, f, indent=2)
        print(f'Saved: {run_dir}/full_results.json')
        print(f'Saved: {save_dir}/full_results.json (latest)')

        alpha_save = ALPHA if isinstance(ALPHA, list) else [float(ALPHA)]
        extra_hp = {
            'feature_dim': FEATURE_DIM,
            'hidden_channels': HIDDEN_CHANNELS,
            'num_layers': NUM_LAYERS,
            'dropout': DROPOUT,
            'search_epochs': SEARCH_EPOCHS,
            'search_batch_size': SEARCH_BATCH_SIZE,
            'search_lr': SEARCH_LR,
            'temperature_start': TEMPERATURE_START,
            'temperature_end': TEMPERATURE_END,
            'finetune_epochs': FINETUNE_EPOCHS,
            'finetune_batch_size': FINETUNE_BATCH_SIZE,
            'finetune_lr': FINETUNE_LR,
            'finetune_patience': FINETUNE_PATIENCE,
            'device': DEVICE,
        }
        save_learnable_ppr_experiment(
            run_dir, dataset_name, enc_type, run_id,
            TELEPORT_VALUES, alpha_save, PPR_EPSILON,
            exp, msp, extra_config=extra_hp)
        print(f'Saved run artifacts: {run_dir}')
print('Done!')